# Osmotic (Bulk) Modulus from Plate-Driven Compression

Post-processing for `compress_slab` runs.  Plots the **osmotic bulk modulus
$K$ as a function of time** over the isotropic compression trajectory.

## Theory

The polymer **osmotic pressure** is the mean normal stress of the network,
$$\Pi(t) = -\tfrac{1}{3}\,\mathrm{tr}\,\boldsymbol{\sigma}^{\mathrm{poly}}(t)
        = -\tfrac{1}{3}\big(\sigma_{xx}^{p}+\sigma_{yy}^{p}+\sigma_{zz}^{p}\big),$$
where $\sigma_{ii}^{p} = \big(\sum_a S_{ii}^{a}\big)/V_{\mathrm{gel}}$ are the
whole-polymer virial normal stresses (LAMMPS `stress/atom NULL`) normalised by
the **Rg-based gel volume** $V_{\mathrm{gel}} = L_x^{Rg} L_y^{Rg} L_z^{Rg}$.
The leading minus sign converts the virial-sign trace into a *positive* osmotic
pressure under compression (`SIGN` toggle below if your convention differs).

The **osmotic bulk modulus** is the volumetric stiffness of the network,
$$K \;=\; -V\left(\frac{\partial \Pi}{\partial V}\right)
      \;=\; -\frac{\partial \Pi}{\partial \ln V}.$$

Along the compression path both $\Pi$ and $V$ are functions of time, so the
**instantaneous** modulus is obtained by the chain rule,
$$K(t) \;=\; -\,V(t)\,\frac{d\Pi/dt}{dV/dt}
        \;=\; -\,\frac{d\Pi/dt}{d\ln V/dt}.$$

$K(t)$ is only well-defined while the gel volume is actually changing (Phase 2
compression); during the frozen-plate equilibration phases $dV/dt \approx 0$ and
the ratio is masked out.  A single plateau value with a confidence interval is
also extracted from a linear $\Pi$-vs-$\ln V$ fit over the compression window.

## Before running: files to copy from the cluster (Pod)

Pod requires a VPN, so this notebook does **not** auto-sync (unlike the Expanse
notebooks — no sync cell here).  Copy the run's output from Pod into your local
`flow_data_local` tree by hand (`scp` while on the VPN, or a mounted drive).
Set `DATANAME`, `INTERACTION`, `NSTEPS`, and `RUN_ID` in the **Config** cell to
match the run; every path is built from those.

**Into** `flow_data_local/bulk/<RUN_ID>/`  *(Pod:* `.../output_files/stress_data/`*)*

| file pattern | Pod subfolder |
|---|---|
| `bulk_modulus_plot_data_<sim_name>.dat` | `stress_data/` |

Plots are written to `flow_data_local/plots/bulk/<RUN_ID>/` (created
automatically by the Config cell).

*`<sim_name>` = `<DATANAME>_<INTERACTION>_<NSTEPS>` — the exact suffix
`compress_slab.lmp` appends to the data file.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
try:
    from scipy.signal import savgol_filter
    _HAS_SAVGOL = True
except ImportError:
    _HAS_SAVGOL = False
from pathlib import Path

# ---- user rcParams (matches triaxial_compression.ipynb / compression_analysis.ipynb) ----
plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False,
})

# ---- colorblind-friendly palettes (from triaxial_compression.ipynb) ----
# Wong (2011) categorical palette for the single-curve plots.
WONG = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73','vermillion':'#D55E00',
        'skyblue':'#56B4E9','yellow':'#F0E442','reddishpurple':'#CC79A7','black':'#000000'}
# 'cividis' is the most CVD-safe sequential map -> used for the time gradient.
EVO_CMAP = 'cividis'
GEL_SHADE = dict(color='0.6', alpha=0.15, zorder=0)   # neutral grey, CVD-safe

print('Imports + style ready  (Savitzky-Golay available:', _HAS_SAVGOL, ')')

## Config

In [ ]:
# ── CONFIG: only change these lines to switch datasets ─────────────────────
# sim_name is the exact suffix compress_slab.lmp appends to the data file:
#     <DATANAME>_<INTERACTION>_<NSTEPS>
DATANAME    = "isolated_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000_with_six_plates"
INTERACTION = "1.0_1.0"                     # epsSS_epsSP
NSTEPS      = 1000000                       # Phase-3 production steps; None -> auto-detect from files
RUN_ID      = "sixplate_rho04_600k_1M"      # local folder label under flow_data_local/{bulk,plots/bulk}
# ───────────────────────────────────────────────────────────────────────────

DATA_DIR = Path("../../flow_data_local/bulk") / RUN_ID
PLOT_DIR = Path("../../flow_data_local/plots/bulk") / RUN_ID
# auto-create the local folders for this run (idempotent; safe to re-run)
for _d in (DATA_DIR, PLOT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print('folders ready:', DATA_DIR, '|', PLOT_DIR)

# Build sim_name.  If NSTEPS is None, auto-detect it from any bulk_modulus_plot_data
# file already in DATA_DIR (after you copy it from Pod); else use it directly.
if NSTEPS is None:
    import re as _re
    _cands = sorted(DATA_DIR.glob(f'bulk_modulus_plot_data_{DATANAME}_{INTERACTION}_*.dat'))
    if not _cands:
        raise ValueError('NSTEPS is None and no bulk_modulus_plot_data_*.dat in DATA_DIR yet '
                         '— set NSTEPS explicitly, or copy the file from Pod first.')
    NSTEPS = int(_re.match(rf'.*_{_re.escape(INTERACTION)}_(\d+)\.dat$', _cands[-1].name).group(1))
    print(f'auto-detected NSTEPS = {NSTEPS} from {_cands[-1].name}')

sim_name  = f'{DATANAME}_{INTERACTION}_{NSTEPS}'
data_file = DATA_DIR / f'bulk_modulus_plot_data_{sim_name}.dat'
print('data file:', data_file)
print('exists   :', data_file.exists())

# ---- analysis parameters ----
dt_lj      = 0.005     # LJ timestep (timestep_prod in compress_slab.lmp)
SIGN       = -1.0      # Π = SIGN * (1/3)tr(σ^poly).  -1 → positive osmotic pressure.
smooth_win = 7         # Savitzky-Golay window (odd, in samples) for dΠ/dt, dV/dt
smooth_ord = 2         # Savitzky-Golay polynomial order
dVdt_frac  = 0.02      # mask K where |dV/dt| < dVdt_frac * max|dV/dt| (equilibration)
ci_level   = 0.95      # confidence level for the plateau-K fit

## Helper Functions

In [ ]:
def read_bulk_modulus_file(filepath):
    '''Read bulk_modulus_plot_data_<sim_name>.dat (fix print output).

    Columns: step  sigP_xx  sigP_yy  sigP_zz  Pi_osm  V_gel_rg  lx_rg  ly_rg  lz_rg
    Skips the '#' title line.  Returns a dict of 1-D numpy arrays.
    '''
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            try:
                rows.append([float(x) for x in line.split()])
            except ValueError:
                pass
    if not rows:
        raise ValueError(f'No numeric data in {filepath}')
    arr = np.array(rows)
    # Drop duplicate steps at phase boundaries (fix print can repeat a step
    # when consecutive run commands share an endpoint); keep the first.
    _, keep = np.unique(arr[:, 0], return_index=True)
    arr = arr[np.sort(keep)]
    cols = ['step', 'sxx', 'syy', 'szz', 'Pi', 'Vgel', 'lx', 'ly', 'lz']
    return {c: arr[:, i] for i, c in enumerate(cols) if i < arr.shape[1]}


def smooth(y, win, order):
    '''Savitzky-Golay smoothing with graceful fallbacks for short series.'''
    n = len(y)
    if n < 5:
        return y.copy()
    w = min(win, n if n % 2 == 1 else n - 1)
    if w % 2 == 0:
        w -= 1
    w = max(w, order + 1 + (1 - (order + 1) % 2))   # keep w > order and odd
    if _HAS_SAVGOL and 3 <= w <= n:
        return savgol_filter(y, w, order)
    k = max(3, w) | 1                               # fallback: centred moving average
    return np.convolve(y, np.ones(k) / k, mode='same')

## Step 1: Load Data

Reads the osmotic-stress / gel-volume time series and derives the osmotic
pressure $\Pi(t)$, gel volume $V(t)$, and elapsed LJ time $t = \text{step}\cdot\Delta t$.

In [ ]:
D = read_bulk_modulus_file(data_file)

step = D['step']
t    = (step - step[0]) * dt_lj            # elapsed time in LJ tau
sxx, syy, szz = D['sxx'], D['syy'], D['szz']
Pi   = SIGN * D['Pi']                       # osmotic pressure (positive convention)
V    = D['Vgel']                            # Rg-based gel volume
lnV  = np.log(V)

# Volumetric strain relative to the start (positive = compression).
eps_vol = 1.0 - V / V[0]

print(f'sim_name: {sim_name}')
print(f'Samples: {len(step)}   steps {int(step[0])}..{int(step[-1])}   '
      f't = 0..{t[-1]:.1f} tau')
print(f'V_gel: {V[0]:.1f} -> {V[-1]:.1f}  (Delta = {100*eps_vol[-1]:.2f}% compression)')
print(f'Pi:    {Pi[0]:.4f} -> {Pi[-1]:.4f}')

## Step 2: Osmotic Stress, Pressure, and Gel Volume vs Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 13), sharex=True, constrained_layout=True)
fig.suptitle(f'Osmotic response: {RUN_ID}', fontsize=16, fontweight='bold')

# (a) polymer normal stresses — Wong categorical palette
ax = axes[0]
ax.plot(t, sxx, '-', color=WONG['blue'],   lw=1.8, label=r'$\sigma^p_{xx}$')
ax.plot(t, syy, '-', color=WONG['orange'], lw=1.8, label=r'$\sigma^p_{yy}$')
ax.plot(t, szz, '-', color=WONG['green'],  lw=1.8, label=r'$\sigma^p_{zz}$')
ax.axhline(0, color='k', ls=':', lw=0.8)
ax.set_ylabel(r'$\sigma^p_{ii}$')
ax.set_title('(a) Polymer normal stresses', fontsize=18)
ax.legend(ncol=3, fontsize=16); ax.grid(alpha=0.3)

# (b) osmotic pressure
ax = axes[1]
ax.plot(t, Pi, '-', color=WONG['vermillion'], lw=2.0)
ax.axhline(0, color='k', ls=':', lw=0.8)
ax.set_ylabel(r'$\Pi$')
ax.set_title(r'(b) Osmotic pressure  $\Pi = -\frac{1}{3}\mathrm{tr}\,\sigma^p$', fontsize=18)
ax.grid(alpha=0.3)

# (c) gel volume
ax = axes[2]
ax.plot(t, V, '-', color=WONG['skyblue'], lw=2.0)
ax.set_ylabel(r'$V_\mathrm{gel}\ (\sigma^3)$')
ax.set_xlabel(r'$t\ (\tau)$')
ax.set_title('(c) Rg-based gel volume', fontsize=18)
ax.grid(alpha=0.3)

plt.savefig(PLOT_DIR / f'osmotic_timeseries_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3: Osmotic Bulk Modulus $K(t)$

Smooth $\Pi(t)$ and $V(t)$, differentiate w.r.t. time, and form
$K(t) = -V\,(d\Pi/dt)/(dV/dt)$.  Points where $|dV/dt|$ is below
`dVdt_frac` of its peak (the frozen-plate equilibration phases) are masked,
since $K$ is indeterminate when the volume is not changing.

In [ ]:
Pi_s = smooth(Pi, smooth_win, smooth_ord)
V_s  = smooth(V,  smooth_win, smooth_ord)

dPi_dt = np.gradient(Pi_s, t)
dV_dt  = np.gradient(V_s,  t)

# K(t) = -V (dPi/dt)/(dV/dt) = -dPi/dlnV
with np.errstate(divide='ignore', invalid='ignore'):
    K_t = -V_s * dPi_dt / dV_dt

# Mask where the gel volume is essentially static (no compression signal).
thresh = dVdt_frac * np.nanmax(np.abs(dV_dt))
active = np.abs(dV_dt) > thresh
K_masked = np.where(active, K_t, np.nan)

n_active = int(np.sum(active))
print(f'Active (compressing) samples: {n_active} / {len(t)}  (|dV/dt| > {thresh:.3g})')
if n_active:
    print(f'K over active window: mean = {np.nanmean(K_masked):.3f}  '
          f'median = {np.nanmedian(K_masked):.3f}')

## Step 4: $K(t)$ — Main Result

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)

# Faint full trace + emphasised active (compressing) window.
ax.plot(t, K_t, '-', color='0.8', lw=1.0, zorder=1,
        label=r'$K(t)$ (all, incl. $dV/dt\!\approx\!0$)')
ax.plot(t, K_masked, '-', color=WONG['green'], lw=2.4, marker='o', ms=4, zorder=3,
        label=r'$K(t)$ (compression window)')

if n_active:
    Kbar = np.nanmean(K_masked)
    ax.axhline(Kbar, color=WONG['vermillion'], ls='--', lw=1.6, zorder=2,
               label=rf'window mean $\bar K = {Kbar:.3f}$')

ax.axhline(0, color='k', ls=':', lw=0.8)
ax.set_xlabel(r'$t\ (\tau)$')
ax.set_ylabel(r'$K = -V\,\dfrac{d\Pi}{dV}$')
ax.set_title(f'Osmotic bulk modulus vs time\n{RUN_ID}', fontsize=16)
ax.legend(fontsize=15); ax.grid(alpha=0.3)

plt.savefig(PLOT_DIR / f'bulk_modulus_vs_time_{sim_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5: Plateau $K$ from a $\Pi$-vs-$\ln V$ Fit

A single, more robust estimate: over the active compression window
$K = -\,d\Pi/d\ln V$ is the negative slope of a linear fit of $\Pi$ against
$\ln V$.  The confidence interval comes from the regression standard error.

In [ ]:
if n_active >= 3:
    xj = lnV[active]
    yj = Pi[active]
    res = stats.linregress(xj, yj)
    K_fit = -res.slope
    df    = n_active - 2
    tcrit = stats.t.ppf((1 + ci_level) / 2, df)
    K_se  = res.stderr
    K_lo, K_hi = K_fit - tcrit * K_se, K_fit + tcrit * K_se

    print('=' * 60)
    print('PLATEAU OSMOTIC BULK MODULUS  (Pi vs lnV linear fit)')
    print('=' * 60)
    print(f'  n points     : {n_active}')
    print(f'  slope dPi/dlnV: {res.slope:.4f}')
    print(f'  R^2          : {res.rvalue**2:.4f}')
    print(f'  K = -slope   : {K_fit:.4f}')
    print(f'  {int(ci_level*100)}% CI      : [{K_lo:.4f}, {K_hi:.4f}]')

    fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
    sc = ax.scatter(xj, yj, c=t[active], cmap=EVO_CMAP, s=48, zorder=3)
    xx = np.linspace(xj.min(), xj.max(), 100)
    ax.plot(xx, res.intercept + res.slope * xx, '--', color=WONG['vermillion'], lw=2,
            label=rf'$K = {K_fit:.3f}$  ($R^2={res.rvalue**2:.3f}$)')
    ax.set_xlabel(r'$\ln V_\mathrm{gel}$')
    ax.set_ylabel(r'$\Pi$')
    ax.set_title('Osmotic pressure vs log gel volume', fontsize=16)
    ax.legend(fontsize=16); ax.grid(alpha=0.3)
    cb = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label(r'$t\ (\tau)$')
    plt.savefig(PLOT_DIR / f'Pi_vs_lnV_fit_{sim_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    K_fit = K_lo = K_hi = np.nan
    print('Not enough active compression samples for a Pi-vs-lnV fit.')

## Optional: Save Data

In [ ]:
out_npz = PLOT_DIR / f'bulk_modulus_{sim_name}.npz'
np.savez(out_npz,
         step=step, t=t,
         sigma_p_xx=sxx, sigma_p_yy=syy, sigma_p_zz=szz,
         Pi=Pi, V_gel=V, lnV=lnV, eps_vol=eps_vol,
         K_t=K_t, K_masked=K_masked, active=active,
         K_fit=K_fit, K_ci_lo=K_lo, K_ci_hi=K_hi, ci_level=ci_level,
         sim_name=sim_name, run_id=RUN_ID)
print(f'Saved: {out_npz}')